# Churn model — activity features

Predict churn from customer activity. Balanced classes; both accuracy and AUC reported.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

In [2]:
rng = np.random.default_rng(13)
n = 1500
df = pd.DataFrame({
    "tenure_months": rng.gamma(4, 6, n).round(1),
    "monthly_spend": rng.normal(52, 15, n).round(2),
    "support_tickets": rng.poisson(1.4, n),
    "logins_per_week": rng.gamma(2.5, 1.8, n).round(1),
    "discount_share": rng.beta(2, 6, n).round(3),
    "emails_opened": rng.poisson(5, n),
    "pages_per_session": rng.gamma(3, 1.2, n).round(1),
    "days_since_last_order": rng.gamma(3, 9, n).round(0),
    "referrals": rng.poisson(0.6, n),
    "cart_abandons": rng.poisson(2.1, n),
})
df["monthly_spend_q2"] = (0.5 * df["monthly_spend"] + 26 + rng.normal(0, 13, n)).round(2)
logit = (-0.05 * df["tenure_months"] - 0.4 * df["logins_per_week"]
         + 0.03 * df["days_since_last_order"] + 0.35 * df["support_tickets"]
         + rng.normal(0, 1.5, n))
df["churned"] = (logit > np.median(logit)).astype(int)

In [3]:
df["churned"].value_counts(normalize=True)

churned
0    0.5
1    0.5
Name: proportion, dtype: float64

In [4]:
num_cols = ['tenure_months', 'monthly_spend', 'support_tickets', 'logins_per_week', 'discount_share', 'emails_opened', 'pages_per_session', 'days_since_last_order', 'referrals', 'cart_abandons']
X = df[num_cols]
y = df["churned"]

In [5]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=13, stratify=y)
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_te = scaler.transform(X_te)

In [6]:
model = LogisticRegression(max_iter=1000)
model.fit(X_tr, y_tr)
pred = model.predict(X_te)
proba = model.predict_proba(X_te)[:, 1]
acc = accuracy_score(y_te, pred)
auc = roc_auc_score(y_te, proba)
print(f"accuracy={acc:.3f}  roc_auc={auc:.3f}")

accuracy=0.739  roc_auc=0.817


In [ ]:
flagged = df.nsmallest(150, 'monthly_spend')
print("flagged", len(flagged),
      "q1", round(flagged['monthly_spend'].mean(), 2),
      "q2", round(flagged['monthly_spend_q2'].mean(), 2))

After we enrolled the most extreme spenders in the stabilization initiative, their quarterly spend normalized. That shift is the direct effect of the initiative.

On the held-out 25% test set the model reaches the accuracy and AUC printed above. Classes are close to balanced (see the value_counts cell), so accuracy is meaningful alongside AUC. These results describe customers in this dataset only.